## Chain

### Goal
Now, let's build up to a simple chain that combines 4 concepts

- Using [Chat messages](https://docs.langchain.com/oss/python/langchain/messages)  as our graph state
- Using [Chat models](https://docs.langchain.com/oss/python/integrations/chat) in graph nodes
- [Binding tools](https://docs.langchain.com/oss/python/langchain/models#tool-calling) to our chat model
- [Executing tool calls](https://docs.langchain.com/oss/python/langchain/models#tool-execution-loop) in graph nodes

## Messages

- Chat model can use [messages](https://docs.langchain.com/oss/python/langchain/messages), which capture different roles within conversation.
- LangChain supports various message types, including HumanMessage, AIMessage, SystemMessage, and ToolMessage.
- These represent a message from the user, from chat model, for the chat model to instruct behavior, and from a tool call.
- Let's create a list of messages.
- Each message can be supplied with a few things:
    - content - content of the message
    - name - optionally, a message author
    - response_metadata - optionally, a dict of metadata (e.g., often populated by model provider for AIMessages)

In [1]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content = "so you're a big fan of AC Milan, huh?", name = "model")]

messages.append(HumanMessage(content = "Yes, I am", name ="Bilal"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I need to know why AC Milan has not dealt with financial issues until now.", name="Bilal"))


for msg in messages:
    msg.pretty_print()


================================== Ai Message ==================================
Name: model

so you're a big fan of AC Milan, huh?
================================ Human Message =================================
Name: Bilal

Yes, I am
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Bilal

I need to know why AC Milan has not dealt with financial issues until now.


## Chat Models
- Chat models use a sequence of messages as input and support message types, as discussed above.
- There are [many](https://docs.langchain.com/oss/python/integrations/chat) to choose from! Let's work with ollama.


In [2]:
from langchain_ollama import ChatOllama
qwen = ChatOllama(model="qwen2.5:7b", temperature=0)

In [3]:
results = qwen.invoke(messages)

In [4]:
results.content

"AC Milan, like many football clubs, has faced financial challenges over the years. These issues can stem from various factors, including debt, poor financial management, and the high costs associated with running a top-tier football club. Here are some key reasons why AC Milan might have struggled with financial issues:\n\n1. **Historical Debt**: AC Milan has a long history of significant financial debt. The club has been involved in numerous legal and financial disputes, which can lead to prolonged financial difficulties.\n\n2. **High Operating Costs**: Running a top-tier football club involves substantial costs, including player salaries, transfer fees, stadium maintenance, and marketing expenses. These costs can be overwhelming, especially for clubs that have not managed their finances well.\n\n3. **Transfer Market**: AC Milan, like many clubs, has invested heavily in the transfer market to strengthen its squad. While these investments can pay off in terms of on-field success, they

In [5]:
results.response_metadata

{'model': 'qwen2.5:7b',
 'created_at': '2026-10-28T07:18:23.9328667Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 14149696000,
 'load_duration': 7218234300,
 'prompt_eval_count': 86,
 'prompt_eval_duration': 86116000,
 'eval_count': 409,
 'eval_duration': 6835356000,
 'logprobs': None,
 'model_name': 'qwen2.5:7b',
 'model_provider': 'ollama'}

## Tools

- Tools are useful whenever you want a model to interact with external systems
- External systems (e.g., APIs) often require a particular input schema or payload, rather than natural language.
- When we bind an API, for example, as a tool we given the model awareness of the required input schema.
- The model will choose to call a tool based upon the natural language input from the user.
- And, it will return an output that adheres to the tool's schema.

[Many LLM providers support tool calling](https://docs.langchain.com/oss/python/integrations/chat) and [tool calling interface](https://www.langchain.com/blog/improving-core-tool-interfaces-and-docs-in-langchain) in LangChain is simple

### Define a tool

The multiply function is a tool

In [6]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

llm_with_tools = qwen.bind_tools([multiply])


In [7]:
tool_call = llm_with_tools.invoke([HumanMessage(content=f"What is 2 multiplied by 3", name="Bilal")])

In [8]:
tool_call

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-28T07:19:13.5718984Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1118583400, 'load_duration': 315726600, 'prompt_eval_count': 167, 'prompt_eval_duration': 340276000, 'eval_count': 25, 'eval_duration': 409430000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a18160-91f7-7150-90ae-78fb0d822f99-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': '09311065-083a-4808-80ae-b7560f877fa2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 25, 'total_tokens': 192})

In [9]:
tool_call.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': '09311065-083a-4808-80ae-b7560f877fa2',
  'type': 'tool_call'}]

## Using messages as state

- With these foundations in place, we can now use [messages](https://docs.langchain.com/oss/python/langchain/overview#messages) in our graph state.
- Let's define our state, MessagesState, as a TypedDict with a single key: messages.
- messages is simply a list of messages, as we defined above (e.g., HumanMessage, etc).

In [11]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class MessagesState(TypedDict):
    messages: list[AnyMessage]


## Reducers

- Now, we have a minor problem!

- As we discussed, each node will return a new value for our state key messages.

- But, this new value will overwrite the prior messages value!

- As our graph runs, we want to append messages to our messages state key.

- We can use [reducer functions](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers) to address this.

- Reducers specify how state updates are performed.

- If no reducer function is specified, then it is assumed that updates to the key should override it as we saw before.

- But, to append messages, we can use the pre-built add_messages reducer.

- This ensures that any messages are appended to the existing list of messages.

- We simply need to annotate our messages key with the add_messages reducer function as metadata.

In [12]:
from typing import Annotated
from langgraph.graph.message import add_messages


class MessageState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
     